In [1]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


In [2]:

# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)



In [3]:


## df = pd.read_csv("cyber_logs.csv")


df = pd.read_csv("cyber_dataset_C.csv")



In [4]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004
...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513


In [5]:

print("Rows:", len(df))
print(df.columns.tolist())


Rows: 1500
['prompt', 'tool_ground_truth', 'tool_predicted', 'pred_svm', 'source', 'row_id']


In [6]:

print("\nGround-truth tools:")
print(df["tool_ground_truth"].value_counts())



Ground-truth tools:
tool_ground_truth
SSHConnect            137
NmapScan              137
BlockIP               137
GetSystemInfo         137
PortScan              136
CheckVulnerability    136
ReadAuthLog           136
CheckFailedLogins     136
ListeningPorts        136
ListProcesses         136
ReadSyslog            136
Name: count, dtype: int64


In [7]:

# Create disagreement label:
# 0 = LLM prediction agrees with annotation
# 1 = LLM prediction disagrees with annotation


In [8]:

df["disagreement"] = (
    df["pred_svm"] != df["tool_ground_truth"]
).astype(int)


In [9]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id,disagreement
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000,0
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001,0
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002,0
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003,0
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004,0
...,...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510,0
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511,0
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512,0
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513,0


In [10]:

print("\nAgreement / disagreement:")
print(df["disagreement"].value_counts())



Agreement / disagreement:
disagreement
0    1378
1     122
Name: count, dtype: int64


In [11]:

# ============================================================
# 2. TF-IDF
# ============================================================




tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=2000
)

X = tfidf.fit_transform(df["prompt"]).toarray()

feature_names = tfidf.get_feature_names_out()

print("\nTF-IDF shape:", X.shape)
print("Number of TF-IDF features:", len(feature_names))




TF-IDF shape: (1500, 2000)
Number of TF-IDF features: 2000


In [12]:

X


array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.1641293, 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ]])

In [13]:

np.set_printoptions(threshold=np.inf)

print(feature_names)


['04' '10' '100' '11' '113' '12' '14' '15' '16' '168' '17' '172' '18' '19'
 '192' '192 168' '198' '198 51' '20' '2001' '2001 db8' '203' '203 113'
 '22' '23' '24' '25' '30' '33' '41' '44' '443' '51' '51 100' '5432' '6379'
 '80' '80 and' '8080' '8443' 'about' 'about the' 'accept'
 'accept connections' 'accepting' 'accepting connections' 'accepts'
 'accepts connections' 'access' 'access attempts' 'access is' 'access to'
 'accessible' 'account' 'account access' 'accounts' 'across' 'active'
 'activity' 'actual' 'actually' 'add' 'address' 'address and' 'address is'
 'address the' 'addresses' 'admin' 'administrative' 'administrator'
 'advisories' 'advisories for' 'advisory' 'affect' 'affected' 'after'
 'after an' 'after its' 'after the' 'after we' 'afternoon' 'afterward'
 'again' 'against' 'against the' 'agent' 'ago' 'alarm' 'alert' 'alias'
 'alive' 'all' 'alone' 'along' 'along with' 'alongside' 'already'
 'already been' 'already know' 'although' 'am' 'an' 'an account'
 'an approved' 'an auth

In [14]:

# ============================================================
# 3. Encode the 11 ground-truth tools
# ============================================================

label_encoder = LabelEncoder()

y_tool = label_encoder.fit_transform(
    df["tool_ground_truth"]
)

tool_names = label_encoder.classes_

print("\nTools:")
for i, tool in enumerate(tool_names):
    print(i, tool)



Tools:
0 BlockIP
1 CheckFailedLogins
2 CheckVulnerability
3 GetSystemInfo
4 ListProcesses
5 ListeningPorts
6 NmapScan
7 PortScan
8 ReadAuthLog
9 ReadSyslog
10 SSHConnect


In [15]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_tool,
    test_size=0.20,
    random_state=SEED,
    stratify=y_tool
)



In [16]:


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)



In [17]:

X_train_t.shape


torch.Size([1200, 2000])

In [18]:

y_train_t.shape


torch.Size([1200])

In [19]:


class ToolNet(nn.Module):

    def __init__(self, input_dim, num_classes):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        return self.net(x)


In [20]:

model_tool = ToolNet(
    input_dim=X.shape[1],
    num_classes=len(tool_names)
)


In [21]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_tool.parameters(),
    lr=0.001
)



In [22]:

EPOCHS = 200

model_tool.train()

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    logits = model_tool(X_train_t)

    loss   = criterion(logits, y_train_t)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(dim=1)

        acc = (
            pred == y_train_t
        ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 2.2742 | Train Accuracy = 0.8425
Epoch  40 | Loss = 1.8071 | Train Accuracy = 0.8983
Epoch  60 | Loss = 0.9507 | Train Accuracy = 0.9667
Epoch  80 | Loss = 0.3130 | Train Accuracy = 0.9942
Epoch 100 | Loss = 0.0996 | Train Accuracy = 0.9975
Epoch 120 | Loss = 0.0405 | Train Accuracy = 0.9992
Epoch 140 | Loss = 0.0222 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0146 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0106 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0081 | Train Accuracy = 1.0000


In [23]:



model_tool.eval()

with torch.no_grad():

    logits = model_tool(X_test_t)

    y_pred = logits.argmax(dim=1).cpu().numpy()


print("\n==========================================")
print("TOOL CLASSIFICATION")
print("==========================================")

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=tool_names
    )
)





TOOL CLASSIFICATION
Test Accuracy: 0.8933333333333333
                    precision    recall  f1-score   support

           BlockIP       1.00      0.93      0.96        28
 CheckFailedLogins       0.96      0.85      0.90        27
CheckVulnerability       1.00      1.00      1.00        27
     GetSystemInfo       0.96      0.89      0.92        27
     ListProcesses       0.86      0.93      0.89        27
    ListeningPorts       0.82      0.67      0.73        27
          NmapScan       0.88      1.00      0.93        28
          PortScan       0.70      0.78      0.74        27
       ReadAuthLog       0.83      0.93      0.88        27
        ReadSyslog       0.92      0.85      0.88        27
        SSHConnect       0.93      1.00      0.97        28

          accuracy                           0.89       300
         macro avg       0.90      0.89      0.89       300
      weighted avg       0.90      0.89      0.89       300



In [24]:


# Maximum observed value for every TF-IDF feature.
#
# NIO will NOT be allowed to produce negative TF-IDF values
# or values larger than those actually observed in the corpus.

feature_max = torch.tensor(
    X.max(axis=0),
    dtype=torch.float32
)



In [25]:

feature_max


tensor([0.3492, 0.4150, 0.2975,  ..., 0.3260, 0.3061, 0.4263])

In [26]:


# ============================================================
# 9. Freeze trained tool model
# ============================================================

for param in model_tool.parameters():

    param.requires_grad = False

model_tool.eval()


ToolNet(
  (net): Sequential(
    (0): Linear(in_features=2000, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=11, bias=True)
  )
)

In [27]:

def nio_tool(
    model,
    target_class,
    feature_max,
    steps=500,
    lr=0.05,
    norm_weight=0.001
):

    # Start near zero inside valid TF-IDF range

    x_opt = torch.rand( 1,  len(feature_max)  )

    x_opt = x_opt * feature_max.unsqueeze(0)

    x_opt.requires_grad_(True)

    optimizer_x = optim.Adam(  [x_opt], lr=lr  )

    target = torch.tensor(  [target_class], dtype=torch.long  )


    for step in range(steps):

        optimizer_x.zero_grad()

        logits = model(x_opt)

        loss_target = criterion( logits, target  )

        # Encourage a relatively small TF-IDF vector
        loss_norm = x_opt.pow(2).mean()

        loss = (  loss_target + norm_weight * loss_norm  )

        loss.backward()
        optimizer_x.step()


        # --------------------------------------------
        # Keep optimized input inside observed
        # TF-IDF feature ranges
        # --------------------------------------------

        with torch.no_grad():

            x_opt.clamp_(min=0.0)

            ## x_opt.copy_( torch.minimum( x_opt, feature_max.unsqueeze(0)  ) )   ## add this back


    return x_opt.detach()




In [28]:


# ============================================================
# 11. Extract top TF-IDF features
# ============================================================

def print_top_features( x_opt, feature_names, top_n=10 ):

    values = (  x_opt.cpu().numpy()[0]  )

    indices = np.argsort(values)[::-1]

    print()

    count = 0

    for idx in indices:

        if values[idx] <= 0:
            continue

        print(
            f"{feature_names[idx]:30s} "
            f"{values[idx]:.4f}"
        )

        count += 1

        if count >= top_n:
            break




In [29]:


# ============================================================
# NIO EXPERIMENT 1
#
#     What TF-IDF features characterize each tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 1")
print("TOOL CHARACTERIZATION")
print("==========================================")

nio_tool_results = {}

for tool_index, tool_name in enumerate(tool_names):

    print("\n------------------------------------------")
    print("TARGET TOOL:", tool_name)
    print("------------------------------------------")

    x_star = nio_tool(
        model=model_tool,
        target_class=tool_index,
        feature_max=feature_max,
        steps=500,
        lr=0.05
    )

    nio_tool_results[tool_name] = x_star

    print_top_features(
        x_star,
        feature_names,
        top_n=10
    )







NIO EXPERIMENT 1
TOOL CHARACTERIZATION

------------------------------------------
TARGET TOOL: BlockIP
------------------------------------------

taking                         0.8744
processor                      0.8527
again                          0.8463
activity                       0.8321
owns                           0.8232
usernames                      0.8125
is consuming                   0.8110
invalid                        0.8102
retired                        0.8077
the listening                  0.8068

------------------------------------------
TARGET TOOL: CheckFailedLogins
------------------------------------------

has the                        0.8624
security                       0.7934
explain                        0.7832
profile                        0.7495
alias                          0.7461
vulnerabilities                0.7430
activity                       0.7332
me which                       0.7309
authenticate                   0.7277
opened  


## Target vs. Pred

* On agreement vs. disagreement label


In [30]:

# ============================================================
# Create disagreement dataset
# ============================================================


# Encode SVM predictions as 11 one-hot features
svm_labels = label_encoder.transform(  df["pred_svm"]  )

svm_onehot = np.eye(len(tool_names))[svm_labels]

# Add to the 2000 TF-IDF features
## X_error = np.hstack((X, svm_onehot))

X_error = X

print(X_error.shape)


(1500, 2000)


In [31]:




########################################

y_error = df["disagreement"].values

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_error,
    y_error,
    test_size=0.20,
    random_state=SEED,
    stratify=y_error
)

X_train_e = torch.tensor( X_train_e, dtype=torch.float32 )

X_test_e = torch.tensor(   X_test_e, dtype=torch.float32 )

y_train_e = torch.tensor( y_train_e, dtype=torch.long    )

y_test_e = torch.tensor(   y_test_e, dtype=torch.long    )



In [32]:

# ============================================================
# 14. Disagreement neural network
# ============================================================

class ErrorNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        return self.net(x)


In [33]:


model_error = ErrorNet(   input_dim=X_error.shape[1]    )

optimizer_error = optim.Adam(  model_error.parameters(),  lr=0.001    )


In [34]:


model_error.train()

for epoch in range(EPOCHS):

    optimizer_error.zero_grad()

    logits = model_error(  X_train_e   ) 

    loss = criterion(  logits,  y_train_e   )

    loss.backward()

    optimizer_error.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(  dim=1  )

        acc = (   pred == y_train_e   ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 0.5078 | Train Accuracy = 0.9183
Epoch  40 | Loss = 0.2570 | Train Accuracy = 0.9183
Epoch  60 | Loss = 0.1742 | Train Accuracy = 0.9183
Epoch  80 | Loss = 0.0812 | Train Accuracy = 0.9775
Epoch 100 | Loss = 0.0308 | Train Accuracy = 0.9992
Epoch 120 | Loss = 0.0077 | Train Accuracy = 1.0000
Epoch 140 | Loss = 0.0025 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0013 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0008 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0005 | Train Accuracy = 1.0000


In [35]:


model_error.eval()

with torch.no_grad():

    logits = model_error(X_test_e)

    pred_error = (  logits.argmax(dim=1).cpu().numpy()  )


print("\n==========================================")
print("DISAGREEMENT CLASSIFICATION")
print("==========================================")

print( "Accuracy:", accuracy_score( y_test_e.numpy(), pred_error ) )

print(
    classification_report(
        y_test_e.numpy(),
        pred_error,
        target_names=[
            "Agreement",
            "Disagreement"
        ]
    )
)




DISAGREEMENT CLASSIFICATION
Accuracy: 0.93
              precision    recall  f1-score   support

   Agreement       0.94      0.99      0.96       276
Disagreement       0.71      0.21      0.32        24

    accuracy                           0.93       300
   macro avg       0.82      0.60      0.64       300
weighted avg       0.92      0.93      0.91       300



In [36]:


for param in model_error.parameters():

    param.requires_grad = False

model_error.eval()



ErrorNet(
  (net): Sequential(
    (0): Linear(in_features=2000, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [37]:


# ============================================================
# NIO EXPERIMENT 2
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION DISAGREEMENT")
print("==========================================")

# Class 1 = disagreement

x_error_star = nio_tool(
    model=model_error,
    target_class=1,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with disagreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)






NIO EXPERIMENT 2
PREDICTION DISAGREEMENT

NIO features associated with disagreement:

the camera                     0.9848
6379                           0.9705
how many                       0.9503
segment                        0.9422
afternoon                      0.9054
investigation                  0.8993
am                             0.8943
audio                          0.8876
the port                       0.8811
this desktop                   0.8684
occupying                      0.8659
integration                    0.8647
connecting                     0.8636
restricted                     0.8633
to this                        0.8598
did the                        0.8575
packet                         0.8536
https                          0.8507
lab vm                         0.8493
need to                        0.8479


In [38]:


# ============================================================
# NIO EXPERIMENT 2.b
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION AGREEMENT")
print("==========================================")

# Class 0 = agreement

x_error_star = nio_tool(
    model=model_error,
    target_class=0,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with agreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)





NIO EXPERIMENT 2
PREDICTION AGREEMENT

NIO features associated with agreement:

link                           0.0085
15                             0.0081
33                             0.0081
know if                        0.0080
written                        0.0078
claimed                        0.0076
match                          0.0074
complete                       0.0074
pid                            0.0073
collect the                    0.0073
should be                      0.0073
is there                       0.0072
redis                          0.0072
linux box                      0.0071
events for                     0.0071
17                             0.0071
udp                            0.0071
authentication records         0.0071
contacting                     0.0070
established                    0.0070


In [39]:

errors = df[df["disagreement"] == 1]

print(
    errors.groupby(
        ["tool_ground_truth", "tool_predicted"]
    ).size().sort_values(ascending=False)
)


tool_ground_truth   tool_predicted    
PortScan            PortScan              15
ReadAuthLog         ReadAuthLog           14
NmapScan            NmapScan              13
CheckFailedLogins   CheckFailedLogins     11
GetSystemInfo       GetSystemInfo         11
ReadSyslog          ReadSyslog             9
ListeningPorts      ListeningPorts         9
CheckVulnerability  CheckVulnerability     8
ListProcesses       ListProcesses          6
BlockIP             BlockIP                4
ListeningPorts      PortScan               4
                    NmapScan               4
PortScan            NmapScan               3
NmapScan            CheckVulnerability     2
ReadSyslog          ReadAuthLog            2
ListProcesses       GetSystemInfo          2
SSHConnect          SSHConnect             2
ListeningPorts      CheckVulnerability     1
ListProcesses       PortScan               1
ReadSyslog          GetSystemInfo          1
dtype: int64


In [41]:

# Deep Gramulator-inspired validation

top_terms = [
    "the camera", "6379", "how many", "segment", "afternoon",
    "investigation", "am", "audio", "the port", "this desktop",
    "occupying", "integration", "connecting", "restricted",
    "to this", "did the", "packet", "https", "lab vm", "need to"
]

X_binary = (X > 0).astype(int)
errors = df["disagreement"].values == 1

print("Term                 Error %    Agreement %    Difference")
print("-" * 62)

for term in top_terms:
    if term not in tfidf.vocabulary_:
        continue

    j = tfidf.vocabulary_[term]
    error_freq = X_binary[errors, j].mean()
    agree_freq = X_binary[~errors, j].mean()

    print(f"{term:20s} {error_freq:8.2%} {agree_freq:13.2%} "
          f"{error_freq - agree_freq:12.2%}")


Term                 Error %    Agreement %    Difference
--------------------------------------------------------------
the camera              0.82%         0.22%        0.60%
6379                    1.64%         0.36%        1.28%
how many                0.82%         0.36%        0.46%
segment                 0.82%         0.44%        0.38%
afternoon               1.64%         0.22%        1.42%
investigation           0.82%         0.44%        0.38%
am                      0.82%         0.36%        0.46%
audio                   0.82%         0.29%        0.53%
the port                3.28%         0.65%        2.63%
this desktop            1.64%         0.36%        1.28%
occupying               1.64%         0.29%        1.35%
integration             0.82%         0.73%        0.09%
connecting              0.82%         0.58%        0.24%
restricted              0.82%         0.36%        0.46%
to this                 1.64%         0.94%        0.70%
did the                 